##### Copyright 2024 Google LLC。

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemma TPU 推斷
此 notebook 示範如何使用 [Flax](https://github.com/google/flax) 和 [Gemma](https://ai.google.dev/gemma)（開放權重大型語言模型 (LLM)）將 Google Colab 的 TPU 用於 @P@P003@P。
<table align="left"> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/.archive/Gemma/[Gemma_1]Inference_on_TPU.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td>
</table>

### 連接到 TPU
- 若要連接到 TPU v2，請點選螢幕右上角的「連接 TPU」按鈕。

現在您可以看到可用的 TPU 裝置：

In [ ]:
import jax

jax.devices()

[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0),
 TpuDevice(id=1, process_index=0, coords=(0,0,0), core_on_chip=1),
 TpuDevice(id=2, process_index=0, coords=(1,0,0), core_on_chip=0),
 TpuDevice(id=3, process_index=0, coords=(1,0,0), core_on_chip=1),
 TpuDevice(id=4, process_index=0, coords=(0,1,0), core_on_chip=0),
 TpuDevice(id=5, process_index=0, coords=(0,1,0), core_on_chip=1),
 TpuDevice(id=6, process_index=0, coords=(1,1,0), core_on_chip=0),
 TpuDevice(id=7, process_index=0, coords=(1,1,0), core_on_chip=1)]

## 安裝

- 要安裝Gemma，您需要使用Python 3.10 或更高版本。
- Google Colab 通常提供Python 3.6 或更高版本作為預設runtime 環境。

In [ ]:
! pip install git+https://github.com/google-deepmind/gemma.git
! pip install --user kaggle

  Cloning https://github.com/google-deepmind/gemma.git to /tmp/pip-req-build-vdzv6aiz
  Running command git clone --filter=blob:none --quiet https://github.com/google-deepmind/gemma.git /tmp/pip-req-build-vdzv6aiz
  Resolved https://github.com/google-deepmind/gemma.git to commit a24194737dcb54b7392091e9ba772aea1cb68ffb
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 下載Gemma 檢查點

在首次使用 [Google Gemma](https://ai.google.dev/gemma) 之前，您必須按照 [Gemma 設定](https://ai.google.dev/gemma/docs/setup) 中的設定說明透過 Kaggle 請求存取模型，或完成以下步驟：
1. 登入 [Kaggle](https://www.kaggle.com) 或建立新的 Kaggle 帳戶（如果您還沒有帳戶）。
1. 前往 [Gemma 型號卡](https://www.kaggle.com/models/google/paligemma/)，然後按一下 **請求存取**。
1. 填寫同意書並接受條款和條件。

若要產生Kaggle API 金鑰，請開啟Kaggle 中的[**設定**頁面](https://www.kaggle.com/settings) 並點選**建立新 token**。這將觸發包含您的 API 憑證的 `kaggle.json` 檔案的下載。
然後，在 Colab 中，選擇左側窗格中的 **Secrets** (🔑) 並新增您的 Kaggle 使用者名稱和 Kaggle API 金鑰。將您的使用者名稱儲存在所需名稱 `KAGGLE_USERNAME` 下，並將 API 金鑰儲存在名稱 `KAGGLE_KEY` 下。

為 Kaggle API 憑證設定環境變數。

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

In [ ]:
import kagglehub

VARIANT = '2b-it' # @param ['2b', '2b-it', '7b', '7b-it'] {type:"string"}
weights_dir = kagglehub.model_download(f'google/gemma/Flax/{VARIANT}')

ckpt_path = os.path.join(weights_dir, VARIANT)
vocab_path = os.path.join(weights_dir, 'tokenizer.model')

In [ ]:
from gemma import params as params_lib
from gemma import sampler as sampler_lib
from gemma import transformer as transformer_lib
import sentencepiece as spm

## 開始使用您的模型進行生成

載入並準備您的 LLM checkpoint 以與 Flax 一起使用。

In [ ]:
# Load parameters
params = params_lib.load_and_format_params(ckpt_path)

載入 tokenizer，您將使用 [SentencePiece](https://github.com/google/sentencepiece) library 建置它。

In [ ]:
vocab = spm.SentencePieceProcessor()
vocab.Load(vocab_path)

True

使用`transformer_lib.TransformerConfig.from_params` 功能自動從checkpoint 載入正確的設定。請注意，由於此版本中未使用tokens，因此詞彙表大小小於輸入嵌入的數量。

In [ ]:
transformer_config=transformer_lib.TransformerConfig.from_params(
    params,
    cache_size=1024  # Number of time steps in the transformer's cache
)
transformer = transformer_lib.Transformer(transformer_config)

最後，在您的模型和 tokenizer 之上建立一個採樣器。

In [ ]:
# Create a sampler with the right param shapes.
sampler = sampler_lib.Sampler(
    transformer=transformer,
    vocab=vocab,
    params=params['transformer'],
)

您已準備好開始採樣！此採樣器使用即時編譯，因此更改輸入形狀會觸發重新編譯，這可能會減慢速度。為了獲得最快、最有效的結果，請保持批次大小一致。

In [ ]:
input_batch = [
    "\n Explain the phenomenon of a solar eclipse.",
  ]

out_data = sampler(
    input_strings=input_batch,
    total_generation_steps=300,
  )

for input_string, out_string in zip(input_batch, out_data.text):
  print(f"Prompt:\n{input_string}\n Answer:\n{out_string}")
  print()

Prompt:

 Explain the phenomenon of a solar eclipse.
 Answer:


A solar eclipse occurs when the Moon passes between the Sun and Earth, casting a shadow on Earth. This phenomenon is caused by the relative positions of the Moon, Sun, and Earth.

**Here's a step-by-step explanation of how a solar eclipse occurs:**

1. **New Moon:** The Moon is positioned between the Sun and Earth, and the Sun's rays are not directly visible from Earth.
2. **Waxing Crescent Phase:** As the Moon orbits the Sun, it gradually moves from the new moon phase to the waxing crescent phase. This means that the illuminated portion of the Moon is gradually increasing.
3. **First Quarter Phase:** When the Moon is at the first quarter phase, half of its face is illuminated.
4. **Waxing Gibbous Phase:** As the Moon continues to orbit the Sun, it moves further away from the Sun, and the illuminated portion of the Moon gradually increases to the waxing gibbous phase. This means that more and more of the Moon is illuminate